# Project: Anomaly Detection in Network Traffic

Detect network anomalies using Isolation Forest, LOF, and Z-score methods.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
print('Libraries loaded')

In [ ]:
np.random.seed(42)
n = 2000
normal = pd.DataFrame({
    'packet_size': np.random.normal(500, 100, n),
    'bytes_sent': np.random.normal(1000, 200, n),
    'connections_sec': np.random.normal(50, 15, n).clip(0),
    'duration_sec': np.random.exponential(30, n),
    'num_ports': np.random.poisson(3, n),
})
anom = pd.DataFrame({
    'packet_size': np.random.normal(1500, 300, 100),
    'bytes_sent': np.random.normal(5000, 1000, 100),
    'connections_sec': np.random.normal(200, 50, 100).clip(0),
    'duration_sec': np.random.exponential(5, 100),
    'num_ports': np.random.poisson(20, 100),
})
df = pd.concat([normal, anom], ignore_index=True)
df['true_label'] = [0]*n + [1]*100
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Total: {len(df)} ({n} normal, 100 anomalies)')
df.head()

In [ ]:
features = ['packet_size','bytes_sent','connections_sec','duration_sec','num_ports']
X = StandardScaler().fit_transform(df[features])
y = df['true_label']

iso = IsolationForest(contamination=0.05, random_state=42)
df['iso_pred'] = (iso.fit_predict(X) == -1).astype(int)
lof = LocalOutlierFactor(contamination=0.05)
df['lof_pred'] = (lof.fit_predict(X) == -1).astype(int)
z_scores = np.abs((df[features] - df[features].mean()) / df[features].std())
df['z_pred'] = (z_scores.max(axis=1) > 3).astype(int)
print('Anomaly detection complete')

In [ ]:
for name, col in [('Isolation Forest','iso_pred'),('LOF','lof_pred'),('Z-Score','z_pred')]:
    cm = confusion_matrix(y, df[col])
    print(f'\n--- {name} ---')
    print(classification_report(y, df[col], target_names=['Normal','Anomaly']))
    print(f'Detection rate: {cm[1,1]/(cm[1,0]+cm[1,1]):.1%} | False positives: {cm[0,1]}')

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_pca[:,0], X_pca[:,1], c=df['true_label'], cmap='coolwarm', alpha=0.5, s=20)
axes[0].set_title('True Labels')
axes[1].scatter(X_pca[:,0], X_pca[:,1], c=df['iso_pred'], cmap='coolwarm', alpha=0.5, s=20)
axes[1].set_title('Isolation Forest Predictions')
plt.tight_layout()
plt.show()

## Summary

- Generated network traffic data with anomalies
- Applied Isolation Forest, LOF, Z-score methods
- Compared detection rates and false positives
- Visualized in PCA space